In [1]:
# =================================================================
# NOTEBOOK: 04_evaluation_and_viz.ipynb
# PHASE 5: EVALUATION AND VISUALIZATION
# GOAL: Final evaluation of the best model (XGBoost), clear reporting of accuracy,
# and visualization of predictions vs. actuals.
# =================================================================

import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 1. File Paths and Setup ---
BASE_PATH = 'C:\\Users\\acer\\OneDrive\\Desktop\\AimlWebforecasting'
DATA_INPUT_PATH = f'{BASE_PATH}\\data\\processed\\features_df.pkl'
ASSETS_PATH = f'{BASE_PATH}\\assets'
RESULTS_PATH = f'{BASE_PATH}\\results'

# Load the necessary assets
FINAL_MODEL_PATH = f'{RESULTS_PATH}\\final_xgb_model.pkl'
PREPROCESSOR_PATH = f'{ASSETS_PATH}\\preprocessor.pkl'
SELECTED_FEATURES_PATH = f'{ASSETS_PATH}\\selected_features.npy'

# Load the best model and processed data files
try:
    final_model = joblib.load(FINAL_MODEL_PATH)
    preprocessor = joblib.load(PREPROCESSOR_PATH)
    selected_features = np.load(SELECTED_FEATURES_PATH, allow_pickle=True).tolist()
    df = pd.read_pickle(DATA_INPUT_PATH)
    print("--- All assets and data loaded successfully ---")
except FileNotFoundError:
    print("Error loading assets. Ensure 03_modeling.ipynb was run successfully and all files exist in 'assets/' and 'results/'.")
    exit()


# --- 2. Data Preparation (Re-create Test Set) ---

# Re-create X and y
y = df['y']
X = df.drop(columns=['y'])

# Split data (must match 03_modeling.ipynb split)
TEST_SIZE = 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, shuffle=False
)

# Process the test set using the fitted preprocessor
X_test_processed = preprocessor.transform(X_test)
feature_names_processed = preprocessor.get_feature_names_out()
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names_processed)

# Filter the test set to include only the selected features
X_test_final = X_test_processed_df[selected_features]


# --- 3. Final Evaluation on Test Set ---

# Make predictions
y_pred = final_model.predict(X_test_final)

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*50)
print("             🎯 FINAL MODEL ACCURACY REPORT 🎯")
print("="*50)

# Print the accuracy percentage in the requested format
print(f"The accuracy for the XGBoost Regressor model is {r2 * 100:.2f} percentage.") 
print(f"Mean Absolute Error (MAE): ${mae:.2f}")

# --- 4. Visualization: Predictions vs. Actuals ---

# Create a DataFrame for visualization using the test set index
results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred}, index=y_test.index)

plt.figure(figsize=(16, 8))
plt.plot(results_df.index, results_df['Actual'], label='Actual Web Traffic (y_test)', color='darkblue', linewidth=2)
plt.plot(results_df.index, results_df['Predicted'], label='Predicted Web Traffic', color='red', linestyle='--', linewidth=1.5)
plt.title(f'XGBoost Forecast vs. Actuals on Test Set (R^2: {r2 * 100:.2f}%)')
plt.xlabel("Date")
plt.ylabel("Web Traffic (Customers)")
plt.legend()
plt.grid(True, which='both', linestyle=':', linewidth=0.5)

# Save the plot
plot_path = f'{RESULTS_PATH}\\04_test_set_forecast.png'
plt.savefig(plot_path)
plt.close()
print(f"\nForecast visualization saved to '{plot_path}'.")


# --- 5. Output Final Results ---

# Save the predictions for potential external reporting
results_df.to_csv(f'{RESULTS_PATH}\\final_predictions.csv')
print(f"Final predictions saved to '{RESULTS_PATH}\\final_predictions.csv'.")

# =================================================================
# END OF NOTEBOOK 04 - Ready for Streamlit App
# =================================================================

--- All assets and data loaded successfully ---

             🎯 FINAL MODEL ACCURACY REPORT 🎯
The accuracy for the XGBoost Regressor model is 93.51 percentage.
Mean Absolute Error (MAE): $147.06

Forecast visualization saved to 'C:\Users\acer\OneDrive\Desktop\AimlWebforecasting\results\04_test_set_forecast.png'.
Final predictions saved to 'C:\Users\acer\OneDrive\Desktop\AimlWebforecasting\results\final_predictions.csv'.
